In [1]:
import pm4py
import pandas as pd

In [ ]:
#we read the log:
sepsis_log=pm4py.read_xes("./Data/sepsis/sepsis.xes")

c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\pm4py\util\dt_parsing\parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(
c:\Users\ccagu\anaconda3\envs\naturalexamples\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 1050/1050 [00:01<00:00, 961.84it/s]


In [ ]:
def build_bag_of_activites_dataset(log):
    #log.groupby("case concept:name")-> group the events of each case to tis case
    #.apply(lambda x: x["concept:name"].value_counts().to_dict()-> to each group get its activities ("concept:name" attribute) and count how many times each one occurs, and convert the result to a dictionary
    act_freq_per_case=log.groupby("case:concept:name").apply(lambda x: x["concept:name"].value_counts().to_dict())
    #Thus act_freq_per_case is a pandas series in which each index is a case and its value is a dictionary where each key is an activity and each value is its frequency
    # case:concept:name   0
    # A                   {"ER registration":5, "Sepsis Triage":3,...}
    # BC                  {"ER registration":1, "Sepsis Triage":1,...}    
    unique_activities_log=log["concept:name"].unique()#get the names of the activites in the whole log
    rows=[]#list to save the activity frequencies of each case
    for case_id, freqs in act_freq_per_case.items():#for each element in the pandas series
        row={act: 0 for act in unique_activities_log}#prepare a dictionary to save the frequencies, by default the frequency is 0
        for act in freqs.keys():#for each activity of the case replace the frequency in row
            row[act]=freqs[act]
        rows.append(list(row.values()))#add this to a list
    #build a dataframe with the previous data
    df_bag_of_activities=pd.DataFrame(data=rows, columns=unique_activities_log)

    df_bag_of_activities["case:concept:name"]=list(act_freq_per_case.index)#create a column with the case id following the order of the pandas series

    return df_bag_of_activities


In [15]:
df_bag_activities_sepsis=build_bag_of_activites_dataset(log=sepsis_log)

In [16]:
df_bag_activities_sepsis

,ER Registration,Leucocytes,CRP,LacticAcid,ER Triage,ER Sepsis Triage,IV Liquid,IV Antibiotics,Admission NC,Release A,Return ER,Admission IC,Release B,Release C,Release D,Release E,case:concept:name
0,1,7,7,1,1,1,1,1,1,1,0,0,0,0,0,0,A
1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,AA
2,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,AAA
3,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,AB
4,1,5,4,1,1,1,1,1,1,1,0,0,0,0,0,0,ABA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1045,1,2,2,1,1,1,1,1,1,1,0,0,0,0,0,0,ZV
1046,1,2,2,1,1,1,1,1,2,1,0,0,0,0,0,0,ZW
1047,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,ZX
1048,1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,ZY


In [ ]:
sepsis_log[sepsis_log["case:concept:name"]=="A"]["concept:name"].value_counts()#double check that everything is correct

Leucocytes          7
CRP                 7
ER Registration     1
LacticAcid          1
ER Triage           1
ER Sepsis Triage    1
IV Liquid           1
IV Antibiotics      1
Admission NC        1
Release A           1
Name: concept:name, dtype: int64

In [ ]:
missing_values = ['nan', 'null', 'None', '']#read the other dataset to configure the order of the cases in the same way later, so that training is not influenced by that
dataset_rules_sepsis=pd.read_csv("./Data/sepsis/mined_sepsis_confidences_SIRS2OrMore.csv",keep_default_na=False, na_values=missing_values)

In [ ]:
df_bag_activities_sepsis=df_bag_activities_sepsis.set_index("case:concept:name")#set the index of the dataframe with bag of activities 

In [ ]:
df_bag_activities_sepsis = df_bag_activities_sepsis.reindex(index=dataset_rules_sepsis['case:concept:name'])#reorder the dataframe based on the order of the declare rule dataset

In [9]:
dataset_rules_sepsis["case:concept:name"]

0         A
1        AA
2        AB
3        AC
4        AD
       ... 
1045    ZJA
1046    ZKA
1047    ZLA
1048    ZMA
1049     ZZ
Name: case:concept:name, Length: 1050, dtype: object

In [ ]:
df_bag_activities_sepsis=df_bag_activities_sepsis.reset_index()#get the case_concept_name as column again

In [ ]:
#get the class of each case
classes=[]
for index, row in df_bag_activities_sepsis.iterrows():
    class_case=dataset_rules_sepsis[dataset_rules_sepsis["case:concept:name"]==row["case:concept:name"]]["Class"].tolist()[0]
    classes.append(class_case)

In [ ]:
df_bag_activities_sepsis["Class"]=classes#create a column with the clases

In [ ]:
df_bag_activities_sepsis.to_csv("./Data/sepsis/sorted_dataset_sepsis_bag_of_activities.csv")#save the dataset

In [77]:
#############################################################################################################################################################################################################

In [ ]:
#the same for the rtfm:
rtfm_log=pm4py.read_xes("./Data/road_traffic/Road_Traffic_Fine_Management_Process.xes")

parsing log, completed traces :: 100%|██████████| 150370/150370 [00:48<00:00, 3111.97it/s]


In [40]:
df_bag_activities_rtfm=build_bag_of_activites_dataset(rtfm_log)

In [41]:
dataset_rules_rtfm=pd.read_csv("./Data/road_traffic/mined_rtfm_relabelled_confidences.csv")

In [42]:
df_bag_activities_rtfm=df_bag_activities_rtfm.set_index("case:concept:name")

In [44]:
df_bag_activities_rtfm=df_bag_activities_rtfm.reindex(index=dataset_rules_rtfm['case:concept:name'])

In [45]:
df_bag_activities_rtfm=df_bag_activities_rtfm.reset_index()

In [46]:
df_bag_activities_rtfm

,case:concept:name,Create Fine,Send Fine,Insert Fine Notification,Add penalty,Send for Credit Collection,Payment,Insert Date Appeal to Prefecture,Send Appeal to Prefecture,Receive Result Appeal from Prefecture,Notify Result Appeal to Offender,Appeal to Judge
0,A1,1,1,0,0,0,0,0,0,0,0,0
1,A100,1,1,1,1,1,0,0,0,0,0,0
2,A10000,1,1,1,1,0,1,0,0,0,0,0
3,A10001,1,1,1,1,0,0,1,1,0,0,0
4,A10004,1,1,1,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
150361,V9995,1,1,1,1,1,0,0,0,0,0,0
150362,V9996,1,1,0,0,0,1,0,0,0,0,0
150363,V9997,1,1,1,1,1,0,0,0,0,0,0
150364,V9998,1,1,1,1,1,0,0,0,0,0,0


In [ ]:
#we use thip map to directly get the class of each case
map_cases_classes = dict(zip(dataset_rules_rtfm["case:concept:name"], dataset_rules_rtfm["Class"]))

df_bag_activities_rtfm["Class"] = df_bag_activities_rtfm["case:concept:name"].map(map_cases_classes)

In [49]:
df_bag_activities_rtfm.head()

,case:concept:name,Create Fine,Send Fine,Insert Fine Notification,Add penalty,Send for Credit Collection,Payment,Insert Date Appeal to Prefecture,Send Appeal to Prefecture,Receive Result Appeal from Prefecture,Notify Result Appeal to Offender,Appeal to Judge,Class
0,A1,1,1,0,0,0,0,0,0,0,0,0,unresolved
1,A100,1,1,1,1,1,0,0,0,0,0,0,collected
2,A10000,1,1,1,1,0,1,0,0,0,0,0,fully_paid
3,A10001,1,1,1,1,0,0,1,1,0,0,0,dismissed
4,A10004,1,1,1,1,1,0,0,0,0,0,0,collected


In [ ]:
#We verify that the clases are corrected

In [50]:
dataset_rules_rtfm[dataset_rules_rtfm["case:concept:name"]=="A1"]["Class"]

0    unresolved
Name: Class, dtype: object

In [51]:
dataset_rules_rtfm[dataset_rules_rtfm["case:concept:name"]=="A100"]["Class"]

1    collected
Name: Class, dtype: object

In [52]:
dataset_rules_rtfm[dataset_rules_rtfm["case:concept:name"]=="A10004"]["Class"]

4    collected
Name: Class, dtype: object

In [53]:
len(df_bag_activities_rtfm)

150366

In [54]:
df_bag_activities_rtfm_cleaned=df_bag_activities_rtfm.dropna(subset=["Class"]).reset_index(drop=True)#drop three cases without class, which were originally considered due to they could belong to more than one class

In [ ]:
df_bag_activities_rtfm_cleaned.to_csv("./Data/road_traffic/sorted_dataset_rtfm_bag_of_activities.csv")#we save the dataset